# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Aleeza-Maryam/Aleeza-ML-Internship-WEEK1/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [2]:
from google.colab import userdata
import duckdb

HF_TOKEN = userdata.get("HF_TOKEN")

print("Token loaded:", HF_TOKEN is not None)

con = duckdb.connect()

try:
    con.execute("DROP SECRET IF EXISTS hf_secret")
except:
    pass

con.execute(f"""
CREATE SECRET hf_secret (
    TYPE HUGGINGFACE,
    TOKEN '{HF_TOKEN}'
)
""")

HF_DATASET = "hf://datasets/FlyRank/internship-warehouse"

print("DuckDB connected")
print("Hugging Face configured")

Token loaded: True
DuckDB connected
Hugging Face configured


In [3]:
con.sql(f"""
SELECT *
FROM read_parquet(
    '{HF_DATASET}/fact_content_daily_performance/month=2026-03/data_0.parquet'
)
LIMIT 5
""").show()

┌─────────────┬─────────────────────────┬──────────────────────────┬────────────────┬────────────────┬────────────────────┬────────────────────┬─────────────────┬────────────┬──────────────────┬───────────────────┬───────────────┬──────────────┬───────────┬──────────────────────┬──────────────────────────┬──────────────────┬─────────────────┬───────────────────┬─────────────────┬───────────────┬─────────────┬────────────┬───────────────┬───────────┬────────────┬───────────┬─────────┬──────────┬───────────────┬─────────┐
│ report_date │     client_hash_id      │     content_hash_id      │ client_has_gsc │ client_has_ga4 │ gsc_data_available │ ga4_data_available │ gsc_impressions │ gsc_clicks │ gsc_sum_position │ gsc_avg_position  │ ga4_pageviews │ ga4_sessions │ ga4_users │ ga4_engaged_sessions │ ga4_total_engagement_sec │ sessions_organic │ sessions_direct │ sessions_referral │ sessions_social │ sessions_paid │ sessions_ai │ ai_chatgpt │ ai_perplexity │ ai_gemini │ ai_copilot │ ai_clau

In [4]:
content_columns = con.sql(f"""
DESCRIBE
SELECT *
FROM read_parquet(
    '{HF_DATASET}/dim_content.parquet'
)
""").df()

print(content_columns["column_name"].tolist())

['client_hash_id', 'content_hash_id', 'keyword_hash_id', 'url_hash_id', 'keyword_char_count', 'keyword_token_count', 'url_char_count', 'content_created_date', 'content_updated_date', 'content_type', 'search_volume', 'competition', 'competition_level', 'cpc', 'main_intent', 'backlinks', 'category_count', 'keyword_created_date', 'provider_used', 'model_used', 'char_count', 'word_count', 'last_optimized_date', 'optimization_eligible_date', 'is_published', 'is_deleted']


In [7]:
march_content = con.sql(f"""
SELECT
    f.client_hash_id,
    f.content_hash_id,
    f.report_date,
    f.gsc_impressions,
    f.gsc_clicks,
    f.gsc_avg_position,
    c.content_updated_date,
    DATE_DIFF(
        'day',
        CAST(c.content_updated_date AS DATE),
        f.report_date
    ) AS days_since_update
FROM read_parquet(
    '{HF_DATASET}/fact_content_daily_performance/month=2026-03/data_0.parquet'
) f
LEFT JOIN read_parquet(
    '{HF_DATASET}/dim_content.parquet'
) c
ON f.client_hash_id = c.client_hash_id
AND f.content_hash_id = c.content_hash_id
""")

print("Rows:", len(march_content))
print("Rows:", len(march_content))
march_content.limit(5).show()

Rows: 9841378


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows: 9841378


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────────────────┬──────────────────────────┬─────────────┬─────────────────┬────────────┬───────────────────┬──────────────────────┬───────────────────┐
│     client_hash_id      │     content_hash_id      │ report_date │ gsc_impressions │ gsc_clicks │ gsc_avg_position  │ content_updated_date │ days_since_update │
│         varchar         │         varchar          │    date     │      int64      │   int64    │      double       │         date         │       int64       │
├─────────────────────────┼──────────────────────────┼─────────────┼─────────────────┼────────────┼───────────────────┼──────────────────────┼───────────────────┤
│ client_73cda7b4e4f265ea │ content_b7e512995f79d5a6 │ 2026-03-01  │              20 │          0 │              3.35 │ 2026-05-18           │               -78 │
│ client_73cda7b4e4f265ea │ content_05597932fe4da067 │ 2026-03-01  │               1 │          0 │               0.0 │ 2026-05-18           │               -78 │
│ client_73cda7b4e4f26

In [8]:
staleness_check = con.sql("""
SELECT
    CASE
        WHEN days_since_update < 90 THEN '0-89 days'
        WHEN days_since_update < 180 THEN '90-179 days'
        WHEN days_since_update < 365 THEN '180-364 days'
        ELSE '365+ days'
    END AS staleness_bucket,
    COUNT(*) AS n,
    ROUND(AVG(gsc_impressions), 2) AS avg_impressions,
    ROUND(AVG(gsc_clicks), 2) AS avg_clicks
FROM march_content
WHERE days_since_update >= 0
GROUP BY 1
ORDER BY
    CASE
        WHEN staleness_bucket = '0-89 days' THEN 1
        WHEN staleness_bucket = '90-179 days' THEN 2
        WHEN staleness_bucket = '180-364 days' THEN 3
        ELSE 4
    END
""")

staleness_check.show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌──────────────────┬────────┬─────────────────┬────────────┐
│ staleness_bucket │   n    │ avg_impressions │ avg_clicks │
│     varchar      │ int64  │     double      │   double   │
├──────────────────┼────────┼─────────────────┼────────────┤
│ 0-89 days        │ 944905 │           35.98 │       0.08 │
│ 90-179 days      │ 134754 │            3.52 │       0.01 │
│ 180-364 days     │  89614 │            0.19 │        0.0 │
└──────────────────┴────────┴─────────────────┴────────────┘



### Signal 1 — Staleness

**Signal:** Days since the content was last updated.

**Why it matters:** Staleness is linked to FlyRank's refresh-flag logic, so it is a relevant signal to audit for a content-refresh baseline.

**Verdict: CONFIRMED**

**Observation:** Older content shows substantially lower average impressions and clicks in this March 2026 slice. Average impressions decrease from 35.98 for content updated within 0–89 days to 0.19 for content that is 180–364 days old. This is a directional observation, not a claim that staleness causes lower traffic.

In [9]:
ctr_position_check = con.sql("""
SELECT
    CASE
        WHEN gsc_avg_position <= 3 THEN '1-3'
        WHEN gsc_avg_position <= 10 THEN '4-10'
        WHEN gsc_avg_position <= 20 THEN '11-20'
        ELSE '21+'
    END AS position_bucket,

    COUNT(*) AS n,

    ROUND(
        AVG(
            CASE
                WHEN gsc_impressions > 0
                THEN CAST(gsc_clicks AS DOUBLE) / gsc_impressions
                ELSE NULL
            END
        ),
        4
    ) AS avg_ctr

FROM march_content

WHERE gsc_avg_position IS NOT NULL
  AND gsc_impressions > 0

GROUP BY 1

ORDER BY
    CASE
        WHEN position_bucket = '1-3' THEN 1
        WHEN position_bucket = '4-10' THEN 2
        WHEN position_bucket = '11-20' THEN 3
        ELSE 4
    END
""")

ctr_position_check.show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────────┬─────────┬─────────┐
│ position_bucket │    n    │ avg_ctr │
│     varchar     │  int64  │ double  │
├─────────────────┼─────────┼─────────┤
│ 1-3             │  727362 │  0.0048 │
│ 4-10            │ 1456122 │  0.0035 │
│ 11-20           │  519223 │  0.0028 │
│ 21+             │  908354 │  0.0013 │
└─────────────────┴─────────┴─────────┘



## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.